# Авито: дообучение bi-encoder (multilingual-e5-base)

**Как запускать:** отдельный ноутбук, акселератор **GPU T4 ×2**, **Internet: On** (один раз скачать модель),
через **Save Version → Save & Run All**. Обучение идёт в фоне, закрытая вкладка ему не мешает. Ориентировочно около часа.

**Входы (Add Input):** вывод ноутбука, где лежат `prep/` и `features/` (у тебя это `avito_kandidates_ranking`).

**Что делает:**
1. Собирает пары «запрос → выбранное объявление» из train, **без запросов CatBoost-train** (иначе косинус на них будет
   неестественно высоким, как с классификатором) и без текстов val
2. Дообучает `intfloat/multilingual-e5-base` с in-batch негативами: для запроса его объявление должно быть ближе,
   чем объявления других запросов в батче
3. Считает векторы всех 189 тыс. объявлений корпуса (полный текст и отдельно заголовок) и запросов CatBoost-train, val и бенчмарка
4. **Проверяет на val:** Recall dense-поиска в гео-пуле (готовая модель против дообученной) и главное —
   **потолок объединённых кандидатов BM25 + dense** против текущих 0.932
5. Сохраняет векторы и модель в `/kaggle/working/encoder/`

Используется только `transformers` и `torch` (предустановлены на Kaggle); модель open-source, работает локально.

### Импорты и параметры

In [1]:
import os, glob, json, math, time, random, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

MODEL_NAME = 'intfloat/multilingual-e5-base'
Q_PREFIX, P_PREFIX = 'query: ', 'passage: '     # префиксы, с которыми обучалась e5: без них качество падает
Q_MAX_LEN, P_MAX_LEN, T_MAX_LEN = 48, 192, 32   # токенов: запрос+фильтр, объявление, только заголовок
PARAMS_CHARS = 300                              # сколько символов параметров объявления брать в текст

MAX_TRAIN_PAIRS = 300_000    # верхняя граница пар для обучения
BATCH_SIZE = 128             # на оба GPU вместе; больше батч = больше негативов для каждого запроса
LR, WARMUP_SHARE, EPOCHS = 2e-5, 0.05, 1
SCALE = 20.0                 # 1 / температура в софтмаксе по косинусам
TIME_BUDGET_MIN = 150        # страховка: обучение остановится, если займёт дольше
ENC_BATCH = 512              # батч при расчёте векторов (без градиентов можно больше)
EVAL_ZERO_SHOT = True        # посчитать качество готовой модели до дообучения (для сравнения)
FREEZE_WORD_EMB = True       # не обучать матрицу словарных эмбеддингов (~70% параметров e5): быстрее, меньше памяти, качество почти не страдает
GEO_RADIUS_KM, K_DENSE = 50, 100
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPU = torch.cuda.device_count()
print(f'устройство: {DEVICE}, GPU: {N_GPU}', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
if N_GPU == 0:
    print('ВНИМАНИЕ: GPU не найден — на CPU обучение займёт очень много времени. Проверь акселератор в настройках.')

def find_dir(filename, roots=('/kaggle/input', '/kaggle/working', '.')):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', filename), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(filename)

PREP_DIR, FEAT_DIR = find_dir('pairs.parquet'), find_dir('features.json')
OUT_DIR = ('/kaggle/working' if os.path.isdir('/kaggle/working') else '.') + '/encoder'
os.makedirs(OUT_DIR, exist_ok=True)
print('prep:', PREP_DIR, '| features:', FEAT_DIR, '| вывод:', OUT_DIR)

устройство: cuda, GPU: 2 ['Tesla T4', 'Tesla T4']
prep: /kaggle/input/notebooks/olegkaran/avito-kandidates-ranking/prep | features: /kaggle/input/notebooks/olegkaran/avito-kandidates-ranking/features | вывод: /kaggle/working/encoder


## 1. Данные и тексты

- **текст запроса** = `query: ` + запрос + фильтр (если есть). Фильтр вида «Вид услуги Красота, здоровье» сильно уточняет смысл
- **текст объявления** = `passage: ` + заголовок + начало параметров + описание (всё обрезается до `P_MAX_LEN` токенов)
- **заголовок отдельно**: для признака `cos_title` (заголовок короткий и точный)

In [2]:
queries = pd.read_parquet(f'{PREP_DIR}/queries.parquet',
                          columns=['qid', 'split', 'query_key', 'query_norm', 'search_infm_params_text',
                                   'search_location_id', 'search_category', 'loc_lat', 'loc_lon'])
pairs = pd.read_parquet(f'{PREP_DIR}/pairs.parquet', columns=['qid', 'item_id', 'split', 'in_corpus'])
import pyarrow.parquet as pq
ITEM_COLS = ['item_id', 'in_corpus', 'title_norm', 'params_norm', 'item_location_id', 'item_category_id', 'lat', 'lon']
has_desc = 'desc_norm' in pq.ParquetFile(f'{PREP_DIR}/items.parquet').schema_arrow.names
items = pd.read_parquet(f'{PREP_DIR}/items.parquet', columns=ITEM_COLS + (['desc_norm'] if has_desc else []))
if not has_desc:   # предобработка v2: описания лежат отдельно в items_text/
    items = items.merge(pd.read_parquet(f'{PREP_DIR}/items_text', columns=['item_id', 'desc_norm']), on='item_id', how='left')
cb_train_qids = set(pd.read_parquet(f'{FEAT_DIR}/train.parquet', columns=['qid'])['qid'].unique())
val_bm25 = pd.read_parquet(f'{FEAT_DIR}/val.parquet', columns=['qid', 'item_id'])
val_qids = set(val_bm25['qid'].unique())

def query_text(row):
    f = (row.search_infm_params_text or '').strip()
    return Q_PREFIX + (row.query_norm or '') + (f' | {f}' if f else '')

queries['q_text'] = [query_text(r) for r in queries.itertuples(index=False)]
items['p_text'] = (P_PREFIX + items['title_norm'].fillna('') + '. ' +
                   items['params_norm'].fillna('').str.slice(0, PARAMS_CHARS) + '. ' + items['desc_norm'].fillna(''))
items['t_text'] = P_PREFIX + items['title_norm'].fillna('')
items = items.drop(columns=['params_norm', 'desc_norm'])
gc.collect()

print(queries[['q_text']].head(3).to_string())
print(items['p_text'].iloc[0][:300])

                                                                                             q_text
0                                                                         query: скупка телевизоров
1                                          query: автоподбор | Рейтинг пользователя 4 звезды и выше
2  query: баня на дровах | Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье
passage: Ремонт/выкуп компьют. и ноутбуков с выездом на дом. Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 82800 Время работы, с 09:30 Время работы, до 23:00 Начальная цена Тип стоимости за услугу Стоимо


## 2. Обучающие пары

Берём пары `split == 'train'`, **исключаем запросы CatBoost-train** (их `qid` из `features/train.parquet`).
Тексты val уже убраны из train при предобработке. Одинаковые пары «текст запроса → объявление» схлопываем.

In [3]:
tr = (pairs[(pairs.split == 'train') & ~pairs.qid.isin(cb_train_qids)][['qid', 'item_id']]
      .merge(queries[['qid', 'q_text']], on='qid')
      .merge(items[['item_id', 'p_text']], on='item_id')
      .drop_duplicates(['q_text', 'item_id']))
if len(tr) > MAX_TRAIN_PAIRS:
    tr = tr.sample(MAX_TRAIN_PAIRS, random_state=SEED)
tr = tr.reset_index(drop=True)
print(f'обучающих пар: {len(tr):,} (уникальных текстов запросов {tr.q_text.nunique():,}, объявлений {tr.item_id.nunique():,})')

обучающих пар: 300,000 (уникальных текстов запросов 81,093, объявлений 244,328)


### Батчи без дубликатов

In-batch негативы: объявления других запросов батча считаются неправильными ответами. Если в батч попадут два одинаковых запроса
или одно объявление дважды, модель будет «наказана» за правильный ответ. Поэтому собираем батчи так, чтобы внутри батча
не повторялись ни тексты запросов, ни объявления.

In [4]:
def make_batches(q_texts, item_ids, batch_size, seed):
    rng = np.random.default_rng(seed)
    pending = list(rng.permutation(len(q_texts)))
    batches = []
    for _ in range(5):                                   # несколько проходов: отложенные примеры идут в следующие батчи
        cur, seen_q, seen_i, leftover = [], set(), set(), []
        for i in pending:
            if q_texts[i] in seen_q or item_ids[i] in seen_i:
                leftover.append(i); continue
            cur.append(i); seen_q.add(q_texts[i]); seen_i.add(item_ids[i])
            if len(cur) == batch_size:
                batches.append(cur); cur, seen_q, seen_i = [], set(), set()
        pending = leftover + cur
        if len(pending) < batch_size:
            break
    rng.shuffle(batches)
    return batches

batches = make_batches(tr.q_text.values, tr.item_id.values, BATCH_SIZE, SEED)
print(f'батчей за эпоху: {len(batches):,} (по {BATCH_SIZE}), неиспользованных пар: {len(tr) - len(batches) * BATCH_SIZE:,}')

батчей за эпоху: 2,317 (по 128), неиспользованных пар: 3,424


## 3. Модель

Кодировщик = трансформер + **mean pooling** по токенам + L2-нормализация (так устроена e5). Смешанная точность fp16
включена **внутри** `forward`: при `DataParallel` на два GPU это обязательно, иначе копии модели на втором GPU считали бы в fp32.

In [5]:
class Encoder(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(name)

    def forward(self, input_ids, attention_mask):
        with torch.autocast('cuda', dtype=torch.float16, enabled=input_ids.is_cuda):
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        emb = (out.float() * m).sum(1) / m.sum(1).clamp(min=1e-6)
        return F.normalize(emb, dim=-1)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = Encoder(MODEL_NAME).to(DEVICE)
model = nn.DataParallel(encoder) if N_GPU > 1 else encoder
if FREEZE_WORD_EMB:
    encoder.bert.get_input_embeddings().weight.requires_grad_(False)
n_all = sum(p.numel() for p in encoder.parameters())
n_train = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f'параметров: {n_all / 1e6:.0f} млн, обучаемых: {n_train / 1e6:.0f} млн')

def tok(texts, max_len):
    b = tokenizer(list(texts), padding=True, truncation=True, max_length=max_len, return_tensors='pt')
    return b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)

@torch.no_grad()
def encode(texts, max_len, batch=ENC_BATCH, desc=''):
    """Тексты -> нормированные векторы float16 (numpy). Сортируем по длине, чтобы меньше тратить на паддинг."""
    model.eval()
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), encoder.bert.config.hidden_size), dtype=np.float16)
    t0 = time.time()
    for s in range(0, len(texts), batch):
        idx = order[s:s + batch]
        out[idx] = model(*tok([texts[i] for i in idx], max_len)).cpu().numpy().astype(np.float16)
    print(f'  {desc}: {len(texts):,} текстов за {time.time() - t0:.0f} с')
    return out

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


параметров: 278 млн, обучаемых: 86 млн


## 4. Функции оценки на val (до и после обучения)

Для каждого val-запроса: гео-пул как в ноутбуке признаков (категория + тот же `location_id` или ближе 50 км),
в пуле берём топ-K по косинусу. Считаем:
- **dense Recall@50 / @100 в пуле**: насколько энкодер сам по себе находит выбранные объявления
- **потолок объединённых кандидатов**: доля релевантных, попавших в «кандидаты BM25 (из val.parquet) ∪ dense топ-100». Сравниваем с текущим потолком BM25

In [6]:
corpus = items[items.in_corpus].reset_index(drop=True)
C_LAT = np.radians(corpus.lat.values.astype('float64')); C_LON = np.radians(corpus.lon.values.astype('float64'))
C_LOC = corpus.item_location_id.values; C_CAT = corpus.item_category_id.values
cid2idx = pd.Series(np.arange(len(corpus)), index=corpus.item_id)

vq = queries[queries.qid.isin(val_qids)].reset_index(drop=True)
val_pos = (pairs[(pairs.split == 'val') & pairs.in_corpus & pairs.item_id.isin(cid2idx.index)]
           .groupby('qid')['item_id'].apply(lambda s: set(cid2idx[s].values)).to_dict())
bm25_cands = val_bm25[val_bm25.item_id.isin(cid2idx.index)].groupby('qid')['item_id'].apply(
    lambda s: set(cid2idx[s].values)).to_dict()

def pool_mask(r):
    cat = (C_CAT == 114) if int(r.search_category) == 114 else np.ones(len(corpus), bool)
    if pd.isna(r.loc_lat):
        return cat
    lat, lon = np.radians(r.loc_lat), np.radians(r.loc_lon)
    a = np.sin((C_LAT - lat) / 2) ** 2 + np.cos(lat) * np.cos(C_LAT) * np.sin((C_LON - lon) / 2) ** 2
    d = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    m = cat & ((C_LOC == r.search_location_id) | (d <= GEO_RADIUS_KM))
    return m if m.any() else cat

VAL_POOLS = [np.flatnonzero(pool_mask(r)) for r in vq.itertuples(index=False)]

def evaluate(item_emb, q_emb, label):
    E = torch.from_numpy(item_emb).to(DEVICE)
    res = {'dense@50': [], 'dense@100': [], 'потолок BM25': [], 'потолок BM25+dense': []}
    for i, r in enumerate(vq.itertuples(index=False)):
        pos = val_pos.get(r.qid)
        if not pos:
            continue
        pool = VAL_POOLS[i]
        s = (E[pool] @ torch.from_numpy(q_emb[i]).to(DEVICE)).float().cpu().numpy()
        top = pool[np.argsort(-s)[:K_DENSE]]
        bm = bm25_cands.get(r.qid, set())
        res['dense@50'].append(len(pos & set(top[:50])) / len(pos))
        res['dense@100'].append(len(pos & set(top)) / len(pos))
        res['потолок BM25'].append(len(pos & bm) / len(pos))
        res['потолок BM25+dense'].append(len(pos & (bm | set(top))) / len(pos))
    del E
    out = {k: float(np.mean(v)) for k, v in res.items()}
    print(f'[{label}] ' + ' | '.join(f'{k}: {v:.4f}' for k, v in out.items()))
    return out

EVAL = {}
if EVAL_ZERO_SHOT:
    print('Готовая модель (до дообучения):')
    EVAL['до дообучения'] = evaluate(encode(corpus.p_text, P_MAX_LEN, desc='корпус'),
                                     encode(vq.q_text, Q_MAX_LEN, desc='val-запросы'), 'до дообучения')

Готовая модель (до дообучения):
  корпус: 189,212 текстов за 412 с
  val-запросы: 3,000 текстов за 1 с
[до дообучения] dense@50: 0.7556 | dense@100: 0.8187 | потолок BM25: 0.9322 | потолок BM25+dense: 0.9608


## 5. Обучение

Лосс: для батча из N пар считаем матрицу косинусов «запросы × объявления» (N×N), правильные ответы на диагонали.
Кросс-энтропия в обе стороны: запрос → объявление и объявление → запрос. AdamW, линейный разогрев 5% и затухание,
градиенты обрезаются по норме 1. Прогресс печатается каждые 100 шагов.

In [7]:
trainable = [p for p in encoder.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)
total_steps = len(batches) * EPOCHS
warmup = max(1, int(total_steps * WARMUP_SHARE))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min((s + 1) / warmup, max(0.0, (total_steps - s) / max(1, total_steps - warmup))))
try:
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
except (AttributeError, TypeError):                     # старые версии torch
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
labels_cache = {}

q_texts, p_texts = tr.q_text.values, tr.p_text.values
t0, step, run_loss, run_n, stopped = time.time(), 0, 0.0, 0, False
for epoch in range(EPOCHS):
    model.train()
    for b in (batches if epoch == 0 else make_batches(q_texts, tr.item_id.values, BATCH_SIZE, SEED + epoch)):
        q = model(*tok(q_texts[b], Q_MAX_LEN))
        p = model(*tok(p_texts[b], P_MAX_LEN))
        scores = SCALE * q @ p.T
        n = scores.shape[0]
        if n not in labels_cache:
            labels_cache[n] = torch.arange(n, device=DEVICE)
        loss = (F.cross_entropy(scores, labels_cache[n]) + F.cross_entropy(scores.T, labels_cache[n])) / 2

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        scaler.step(opt); scaler.update(); sched.step()

        step += 1
        run_loss += loss.item(); run_n += 1
        if step % 100 == 0 or step == total_steps:
            el = time.time() - t0
            print(f'шаг {step:,}/{total_steps:,} | loss {run_loss / run_n:.4f} | {el / 60:.1f} мин, '
                  f'осталось ~{el / step * (total_steps - step) / 60:.0f} мин', flush=True)
            run_loss, run_n = 0.0, 0
        if time.time() - t0 > TIME_BUDGET_MIN * 60:
            print(f'достигнут лимит времени {TIME_BUDGET_MIN} мин, остановка на шаге {step}')
            stopped = True
            break
    if stopped:
        break
print(f'обучение: {step:,} шагов за {(time.time() - t0) / 60:.1f} мин')
del opt, scaler; gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

шаг 100/2,317 | loss 1.6387 | 2.0 мин, осталось ~43 мин
шаг 200/2,317 | loss 0.7843 | 3.9 мин, осталось ~41 мин
шаг 300/2,317 | loss 0.7104 | 5.8 мин, осталось ~39 мин
шаг 400/2,317 | loss 0.6855 | 7.8 мин, осталось ~37 мин
шаг 500/2,317 | loss 0.6691 | 9.7 мин, осталось ~35 мин
шаг 600/2,317 | loss 0.6729 | 11.7 мин, осталось ~33 мин
шаг 700/2,317 | loss 0.6450 | 13.6 мин, осталось ~31 мин
шаг 800/2,317 | loss 0.6408 | 15.6 мин, осталось ~30 мин
шаг 900/2,317 | loss 0.6348 | 17.5 мин, осталось ~28 мин
шаг 1,000/2,317 | loss 0.6308 | 19.5 мин, осталось ~26 мин
шаг 1,100/2,317 | loss 0.6250 | 21.4 мин, осталось ~24 мин
шаг 1,200/2,317 | loss 0.6115 | 23.3 мин, осталось ~22 мин
шаг 1,300/2,317 | loss 0.6168 | 25.3 мин, осталось ~20 мин
шаг 1,400/2,317 | loss 0.5899 | 27.2 мин, осталось ~18 мин
шаг 1,500/2,317 | loss 0.5998 | 29.2 мин, осталось ~16 мин
шаг 1,600/2,317 | loss 0.6035 | 31.1 мин, осталось ~14 мин
шаг 1,700/2,317 | loss 0.5777 | 33.1 мин, осталось ~12 мин
шаг 1,800/2,317 | lo

## 6. Векторы и оценка дообученной модели

Считаем векторы корпуса (полный текст и заголовок) и запросов CatBoost-train, val и бенчмарка. Затем та же оценка на val.

In [8]:
item_emb = encode(corpus.p_text, P_MAX_LEN, desc='корпус: полный текст')
item_title_emb = encode(corpus.t_text, T_MAX_LEN, desc='корпус: заголовки')

need_q = queries[queries.qid.isin(cb_train_qids | val_qids) | (queries.split == 'bench')].reset_index(drop=True)
query_emb = encode(need_q.q_text, Q_MAX_LEN, desc='запросы (CatBoost-train, val, бенчмарк)')

q_pos = pd.Series(np.arange(len(need_q)), index=need_q.qid)
EVAL['после дообучения'] = evaluate(item_emb, query_emb[q_pos[vq.qid].values], 'после дообучения')
display(pd.DataFrame(EVAL).T.style.format('{:.4f}'))

  корпус: полный текст: 189,212 текстов за 421 с
  корпус: заголовки: 189,212 текстов за 68 с
  запросы (CatBoost-train, val, бенчмарк): 20,452 текстов за 8 с
[после дообучения] dense@50: 0.8359 | dense@100: 0.8846 | потолок BM25: 0.9322 | потолок BM25+dense: 0.9673


,dense@50,dense@100,потолок BM25,потолок BM25+dense
до дообучения,0.7556,0.8187,0.9322,0.9608
после дообучения,0.8359,0.8846,0.9322,0.9673


### Как читать таблицу

- **потолок BM25** — текущая доля релевантных среди кандидатов (~0.93)
- **потолок BM25+dense** — то же, если добавить топ-100 энкодера. **Разница между ними — сколько из тех 4.7% «в пуле, но не по словам» энкодер возвращает**
- **dense@50** можно сравнить с BM25@50 в пуле (0.835): если энкодер в одиночку близок или выше, он сильный самостоятельный сигнал

## 7. Сохранение векторов и модели

In [9]:
np.save(f'{OUT_DIR}/item_emb.npy', item_emb)
np.save(f'{OUT_DIR}/item_title_emb.npy', item_title_emb)
np.save(f'{OUT_DIR}/query_emb.npy', query_emb)
pd.DataFrame({'item_id': corpus.item_id}).to_parquet(f'{OUT_DIR}/item_ids.parquet', index=False)
pd.DataFrame({'qid': need_q.qid}).to_parquet(f'{OUT_DIR}/query_ids.parquet', index=False)

encoder.bert.save_pretrained(f'{OUT_DIR}/model')
tokenizer.save_pretrained(f'{OUT_DIR}/model')
json.dump({'base_model': MODEL_NAME, 'q_prefix': Q_PREFIX, 'p_prefix': P_PREFIX,
           'q_max_len': Q_MAX_LEN, 'p_max_len': P_MAX_LEN, 't_max_len': T_MAX_LEN, 'params_chars': PARAMS_CHARS,
           'train_pairs': len(tr), 'steps': step, 'batch_size': BATCH_SIZE, 'lr': LR, 'eval': EVAL},
          open(f'{OUT_DIR}/meta.json', 'w'), ensure_ascii=False, indent=2)
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    size = sum(os.path.getsize(os.path.join(p, x)) for x in os.listdir(p)) if os.path.isdir(p) else os.path.getsize(p)
    print(f'{f:<22} {size / 2**20:8.1f} МБ')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

item_emb.npy              277.2 МБ
item_ids.parquet            3.1 МБ
item_title_emb.npy        277.2 МБ
meta.json                   0.0 МБ
model                    1076.7 МБ
query_emb.npy              30.0 МБ
query_ids.parquet           0.4 МБ
